# Visualização Amazon Data e Bioflore Data

Este notebook visualiza:
1. Visualização reduzida da ortoimagem completa do Amazon Data
2. Visualização das 3 imagens completas do bioflore data (BigPlot 03, 07, 11) lado a lado com subtítulos

## 1. Imports e Setup

In [ ]:
import sys
import os
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from typing import Optional

# Adicionar o diretório raiz ao path
current_dir = Path(os.getcwd()).resolve()
if current_dir.name == 'exploration_notebooks':
    PROJECT_ROOT = current_dir.parent
elif 'exploration_notebooks' in str(current_dir):
    idx = str(current_dir).find('exploration_notebooks')
    PROJECT_ROOT = Path(str(current_dir)[:idx]).parent if idx > 0 else current_dir.parent.parent
else:
    PROJECT_ROOT = Path("/home/luizluz/Documentos/multi-task-fcn")

sys.path.insert(0, str(PROJECT_ROOT))

from src.io_operations import load_image
from utils import fast_imshow
import cv2
from typing import Optional

# Função auxiliar para visualizar imagens RGB com interpolação linear
def fast_imshow_rgb(image: np.ndarray, ax: Optional[plt.Axes] = None, scale: float = 1/16, **kwargs):
    """
    Mostra uma imagem RGB rapidamente fazendo resize com interpolação linear.
    Melhor para imagens RGB do que fast_imshow que usa INTER_NEAREST.
    """
    if ax is None:
        fig, ax = plt.subplots(figsize=(10, 10))
    
    # Handle boolean images
    if image.dtype == bool:
        image = image.astype(np.uint8) * 255
    
    # Resize if image is large enough
    h, w = image.shape[:2]
    new_h, new_w = int(h * scale), int(w * scale)
    
    if new_h > 0 and new_w > 0:
        # Use INTER_LINEAR for RGB images (better than INTER_NEAREST)
        if image.ndim == 3 and image.shape[2] == 3:
            # RGB image - use linear interpolation
            image = cv2.resize(image, (new_w, new_h), interpolation=cv2.INTER_LANCZOS4)
        else:
            # Single channel or other - use nearest
            image = cv2.resize(image, (new_w, new_h), interpolation=cv2.INTER_NEAREST)
    
    ax.imshow(image, **kwargs)
    return ax

# Criar pasta figures se não existir
FIGURES_DIR = PROJECT_ROOT / "exploration_notebooks" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"FIGURES_DIR: {FIGURES_DIR}")

## 2. Visualização Amazon Data

In [ ]:
# Caminho para a ortoimagem do Amazon
amazon_ortho_path = PROJECT_ROOT / "amazon_data" / "amazon_input_data" / "orthoimage" / "NOV_2017_FINAL_004.tif"

print(f"Carregando imagem do Amazon: {amazon_ortho_path}")
amazon_image = load_image(str(amazon_ortho_path))

print(f"Shape da imagem: {amazon_image.shape}")
print(f"Dtype: {amazon_image.dtype}")

# Converter de (bands, height, width) para (height, width, bands) se necessário
if amazon_image.ndim == 3 and amazon_image.shape[0] <= 4:
    # Pegar apenas as 3 primeiras bandas (RGB)
    amazon_rgb = amazon_image[:3, :, :]
    # Converter para (height, width, bands)
    amazon_rgb = np.moveaxis(amazon_rgb, 0, -1)
    
    # Normalizar para visualização usando quantile para evitar bordas pretas
    # Usar quantile(0.99) ao invés de max() para ser mais robusto a outliers
    if amazon_rgb.dtype != np.uint8:
        amazon_rgb = amazon_rgb.astype(np.float32)
        # Normalizar cada banda separadamente usando quantile
        for band_idx in range(3):
            band = amazon_rgb[:, :, band_idx]
            # Usar quantile 0.99 para evitar outliers
            q99 = np.quantile(band[band > 0] if np.any(band > 0) else band, 0.99)
            q01 = np.quantile(band[band > 0] if np.any(band > 0) else band, 0.01)
            if q99 > q01:
                # Normalizar e clipar entre 0 e 255
                band_normalized = np.clip((band - q01) / (q99 - q01) * 255, 0, 255)
                amazon_rgb[:, :, band_idx] = band_normalized
        amazon_rgb = amazon_rgb.astype(np.uint8)
    elif amazon_rgb.max() > 255:
        # Se já está em uint8 mas tem valores > 255, normalizar
        amazon_rgb = amazon_rgb.astype(np.float32)
        for band_idx in range(3):
            band = amazon_rgb[:, :, band_idx]
            q99 = np.quantile(band[band > 0] if np.any(band > 0) else band, 0.99)
            q01 = np.quantile(band[band > 0] if np.any(band > 0) else band, 0.01)
            if q99 > q01:
                band_normalized = np.clip((band - q01) / (q99 - q01) * 255, 0, 255)
                amazon_rgb[:, :, band_idx] = band_normalized
        amazon_rgb = amazon_rgb.astype(np.uint8)
else:
    amazon_rgb = amazon_image


In [ ]:
amazon_rgb[np.all(amazon_rgb==0, axis=-1)] = 255

In [ ]:

print(f"Shape para visualização: {amazon_rgb.shape}")

# Visualizar usando fast_imshow, removendo título e bordas
fig, ax = plt.subplots(figsize=(12, 12))
fast_imshow(amazon_rgb, ax=ax)
ax.axis('off')

# Salvar sem título ou bordas
pdf_path = FIGURES_DIR / "amazon_orthoimage.png"
plt.savefig(pdf_path, bbox_inches='tight', pad_inches=0, dpi=120)
print(f"Plot salvo em: {pdf_path}")

plt.show()

## 3. Visualização Bioflore Data - 3 Regiões

In [ ]:
# Caminhos para as 3 imagens do bioflore
bioflore_paths = [
    PROJECT_ROOT / "matematica_industria_data" / "raw" / "geotiffs" / "Mosaic_BigPlot_03.tif",
    PROJECT_ROOT / "matematica_industria_data" / "raw" / "geotiffs" / "Mosaic_BigPlot_07.tif",
    PROJECT_ROOT / "matematica_industria_data" / "raw" / "geotiffs" / "Mosaic_BigPlot_11.tif"
]

region_titles = ["Region 1", "Region 2", "Region 3"]

# Carregar as 3 imagens
bioflore_images = []
for i, path in enumerate(bioflore_paths):
    print(f"Carregando {region_titles[i]}: {path}")
    img = load_image(str(path))
    
    if i == 0:
        img[np.all(img == 0, axis=-1)] = 255
        
    print(f"  Shape: {img.shape}, Dtype: {img.dtype}")
    
    # Converter de (bands, height, width) para (height, width, bands) se necessário
    if img.ndim == 3 and img.shape[0] <= 4:
        # Pegar apenas as 3 primeiras bandas (RGB)
        img_rgb = img[:3, :, :]
        # Converter para (height, width, bands)
        img_rgb = np.moveaxis(img_rgb, 0, -1)
        
        # Normalizar para visualização usando quantile para evitar bordas pretas
        # Usar quantile(0.99) ao invés de max() para ser mais robusto a outliers
        if img_rgb.dtype != np.uint8:
            img_rgb = img_rgb.astype(np.float32)
            # Normalizar cada banda separadamente usando quantile
            for band_idx in range(3):
                band = img_rgb[:, :, band_idx]
                # Usar quantile 0.99 para evitar outliers
                q99 = np.quantile(band[band > 0] if np.any(band > 0) else band, 0.99)
                q01 = np.quantile(band[band > 0] if np.any(band > 0) else band, 0.01)
                if q99 > q01:
                    # Normalizar e clipar entre 0 e 255
                    band_normalized = np.clip((band - q01) / (q99 - q01) * 255, 0, 255)
                    img_rgb[:, :, band_idx] = band_normalized
            img_rgb = img_rgb.astype(np.uint8)
        elif img_rgb.max() > 255:
            # Se já está em uint8 mas tem valores > 255, normalizar
            img_rgb = img_rgb.astype(np.float32)
            for band_idx in range(3):
                band = img_rgb[:, :, band_idx]
                q99 = np.quantile(band[band > 0] if np.any(band > 0) else band, 0.99)
                q01 = np.quantile(band[band > 0] if np.any(band > 0) else band, 0.01)
                if q99 > q01:
                    band_normalized = np.clip((band - q01) / (q99 - q01) * 255, 0, 255)
                    img_rgb[:, :, band_idx] = band_normalized
            img_rgb = img_rgb.astype(np.uint8)
    else:
        img_rgb = img
    
    bioflore_images.append(img_rgb)
    print(f"  Shape para visualização: {img_rgb.shape}\n")


In [ ]:
bioflore_images[0][np.all((bioflore_images[0]==0), axis=-1)] = 255

In [ ]:

# Criar figura com 3 subplots horizontais
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Visualizar cada imagem usando fast_imshow_rgb (melhor para RGB)
# Melhorar a visualização para evitar bordas pretas
for i, (img, title) in enumerate(zip(bioflore_images, region_titles)):
    # Garantir que a imagem está no formato correto e não tem valores inválidos
    img_clean = np.nan_to_num(img, nan=0.0, posinf=255.0, neginf=0.0)
    img_clean = np.clip(img_clean, 0, 255).astype(np.uint8)
    
    # Usar fast_imshow_rgb que usa interpolação linear (melhor para RGB)
    fast_imshow_rgb(img_clean, ax=axes[i])
    axes[i].set_title(title, fontsize=14)
    axes[i].axis('off')
    # Garantir que o aspect ratio está correto para evitar distorções
    axes[i].set_aspect('auto')

plt.tight_layout()

# Salvar como PDF
pdf_path = FIGURES_DIR / "bioflore_regions.png"
plt.savefig(pdf_path, bbox_inches='tight', dpi=120)
print(f"Plot salvo em: {pdf_path}")

plt.show()